In [1]:
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset

from torchvision import transforms
from torchvision.transforms import ToTensor
from torchvision import datasets

from torch.optim import SGD
from torch.nn import CrossEntropyLoss

from sklearn.model_selection import StratifiedShuffleSplit
import sys
import os

# PCB Defect Detection - Initial Exploration

Starting with a classification approach. Plan is to build up from simple baselines to more complex CNN architectures. Let's see how the data looks first.

In [2]:
!nvidia-smi

Tue Apr  7 03:42:14 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 595.58.03              Driver Version: 595.58.03      CUDA Version: 13.2     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4050 ...    Off |   00000000:01:00.0 Off |                  N/A |
| N/A   46C    P0             14W /   80W |      26MiB /   6141MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [4]:
device

'cuda'

Good - GPU is available (RTX 4050). 6GB VRAM should be plenty for 224x224 images. Now let's load the PCB dataset and check what we're working with.

In [5]:
sys.path.append(os.path.abspath(os.path.join('..')))
import config

data_dir = config.IMAGES_DIR

print(f"Loading images from: {data_dir}")

data_transforms = transforms.Compose([
    transforms.Resize((224,224)),
])

pcb_dataset = datasets.ImageFolder(root=data_dir,
                                   transform=data_transforms)


# Check the classes found
print(f"Detected classes: {pcb_dataset.classes}")

Loading images from: /mnt/WorkSpace/Repos/PCB-Defect-detection-using-CNN/Data/1/PCB_DATASET/images
Detected classes: ['Missing_hole', 'Mouse_bite', 'Open_circuit', 'Short', 'Spur', 'Spurious_copper']


In [6]:
len(pcb_dataset)

693

In [7]:
#Image 1

pcb_dataset[0]

(<PIL.Image.Image image mode=RGB size=224x224>, 0)

693 images across 6 defect types. That's not a lot of data - will definitely need data augmentation. Noticed the dataset is pretty balanced (~115-116 per class). Let's check one sample to see the image quality.

In [8]:
image, label = pcb_dataset[0]    
label

0

In [9]:
labels = pcb_dataset.targets
for i in range(10):
    print(labels[i])

0
0
0
0
0
0
0
0
0
0


In [10]:
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

train_idx, test_idx = next(
    sss.split(range(len(pcb_dataset)), labels)
)


Standard stratified split (80/20) to maintain class balance. Creating custom subset class so we can apply different transforms for train vs test - need augmentation for the training set only.

In [11]:
len(train_idx), len(test_idx)

(554, 139)

In [12]:
from torch.utils.data import Subset

train_dataset = Subset(pcb_dataset, train_idx)
test_dataset = Subset(pcb_dataset, test_idx)

In [13]:
len(train_dataset), len(test_dataset)

(554, 139)

In [14]:
class CustomSubset(Dataset):
    def __init__(self, subset, transform):
        self.subset = subset
        self.transform = transform

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, idx):
        image, label = self.subset[idx]
        
        if self.transform:
            image = self.transform(image)
        
        return image, label

Adding random rotation (±15°) and Gaussian blur for augmentation. PCB defects should be somewhat rotation-invariant, but not too much since boards have fixed orientations. Blur helps with image quality variations from the manufacturing line.

In [15]:
train_transforms = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomRotation(15),
    transforms.GaussianBlur(3),
    transforms.ToTensor(),
    ]
)

test_transforms = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor()
    ]
)

In [16]:
train_dataset = CustomSubset(train_dataset, transform=train_transforms)
test_dataset  = CustomSubset(test_dataset, transform=test_transforms)


In [17]:
print(train_dataset)

In [18]:
!nvidia-smi

Tue Apr  7 03:42:15 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 595.58.03              Driver Version: 595.58.03      CUDA Version: 13.2     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4050 ...    Off |   00000000:01:00.0 Off |                  N/A |
| N/A   45C    P4              6W /   80W |      29MiB /   6141MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# Building a Data Loader

In [19]:
BATCH_SIZE = 16

In [ ]:
train_dataloader = DataLoader(dataset=train_dataset,
                              batch_size=BATCH_SIZE,
                              shuffle=True,
                            )

test_dataloader = DataLoader(dataset=test_dataset,
                              batch_size=BATCH_SIZE,
                              shuffle=False,
]=-[09p87uy65trgdfsccpl;okijuyhgtfrdesxza?]                              )


In [21]:
train_dataloader, test_dataloader

(<torch.utils.data.dataloader.DataLoader at 0x7fdce0681110>,
 <torch.utils.data.dataloader.DataLoader at 0x7fdce2490dd0>)

In [22]:
print(f"Length of Train Dataloader: {len(train_dataloader)} and shape: {train_dataloader.batch_size}")
print(f"Length of Test Dataloader: {len(test_dataloader)} and shape: {test_dataloader.batch_size}")

Length of Train Dataloader: 35 and shape: 16
Length of Test Dataloader: 9 and shape: 16


In [23]:
!nvidia-smi

Tue Apr  7 03:42:16 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 595.58.03              Driver Version: 595.58.03      CUDA Version: 13.2     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4050 ...    Off |   00000000:01:00.0 Off |                  N/A |
| N/A   45C    P4              6W /   80W |      29MiB /   6141MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [24]:
train_features_batch, train_labels_batch = next(iter(train_dataloader))



In [25]:
class_name = pcb_dataset.classes
class_name

['Missing_hole',
 'Mouse_bite',
 'Open_circuit',
 'Short',
 'Spur',
 'Spurious_copper']

In [26]:
train_labels_batch[0]

tensor(3)

In [27]:
pcb_dataset.class_to_idx

{'Missing_hole': 0,
 'Mouse_bite': 1,
 'Open_circuit': 2,
 'Short': 3,
 'Spur': 4,
 'Spurious_copper': 5}

In [28]:
train_features_batch.shape

torch.Size([16, 3, 224, 224])

## Baseline: Linear Model (No Activation)

Starting with the simplest possible model - just flatten and linear layers. This will be our "lower bound" for performance. If this works at all, we know the problem is solvable. Expecting terrible accuracy since there's no spatial feature learning.

In [29]:
class LinearBaseline(nn.Module):
    def __init__(self, in_features, hidden_units, out_features) -> None:
        super().__init__()

        self.layer_stack = nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_features=in_features,
                      out_features=hidden_units),
            nn.Linear(in_features=hidden_units,
                      out_features=out_features)
        )

    
    def forward(self, X):
        return self.layer_stack(X)

In [30]:
model_0 = LinearBaseline(in_features=3*224*224,
                            hidden_units=50,
                            out_features=len(pcb_dataset.classes)).to(device=device)

model_0

LinearBaseline(
  (layer_stack): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=150528, out_features=50, bias=True)
    (2): Linear(in_features=50, out_features=6, bias=True)
  )
)

In [31]:
loss_fn = CrossEntropyLoss()
optimizer = SGD(params=model_0.parameters(),
                lr=0.01)

In [32]:
from timeit import default_timer as timer

def print_train_time(start, end, device):
    total_time = end-start
    print(f"Train time on {device}: {total_time:.2f}")

In [33]:
from tqdm.auto import tqdm

In [34]:
Epochs = 10 

train_time_on_gpu = timer()
for epoch in tqdm(range(Epochs)):
    print(f"Epoch: {epoch + 1} \n-----------")
    epoch_loss = 0
    
    for batch, (X ,y) in enumerate(train_dataloader):
        model_0.train()

        X = X.to(device)
        y = y.to(device)

        y_pred = model_0(X)

        loss = loss_fn(y_pred, y)
        epoch_loss += loss.item()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if batch % 15 == 0 or batch == len(train_dataloader):
            print(f"  Batch {batch}/{len(train_dataloader)} - Loss: {loss.item():.4f}")
    
    print(f"Avg loss: {epoch_loss/len(train_dataloader):.4f}\n")

train_time_off_gpu = timer()
print_train_time(train_time_on_gpu, train_time_off_gpu, device)

  0%|          | 0/10 [00:00<?, ?it/s]

Epoch: 1 
-----------
  Batch 0/35 - Loss: 1.8209
  Batch 15/35 - Loss: 2.0182
  Batch 30/35 - Loss: 1.8464
Avg loss: 3.1392

Epoch: 2 
-----------
  Batch 0/35 - Loss: 2.1886
  Batch 15/35 - Loss: 2.3020
  Batch 30/35 - Loss: 1.8994
Avg loss: 1.9887

Epoch: 3 
-----------
  Batch 0/35 - Loss: 2.1675
  Batch 15/35 - Loss: 1.8938
  Batch 30/35 - Loss: 1.8674
Avg loss: 1.8486

Epoch: 4 
-----------
  Batch 0/35 - Loss: 1.7159
  Batch 15/35 - Loss: 1.9144
  Batch 30/35 - Loss: 1.8468
Avg loss: 1.8406

Epoch: 5 
-----------
  Batch 0/35 - Loss: 1.8790
  Batch 15/35 - Loss: 1.7555
  Batch 30/35 - Loss: 1.9078
Avg loss: 1.8301

Epoch: 6 
-----------
  Batch 0/35 - Loss: 1.8087
  Batch 15/35 - Loss: 1.8968
  Batch 30/35 - Loss: 1.8363
Avg loss: 1.8078

Epoch: 7 
-----------
  Batch 0/35 - Loss: 1.7834
  Batch 15/35 - Loss: 1.8402
  Batch 30/35 - Loss: 1.7780
Avg loss: 1.8123

Epoch: 8 
-----------
  Batch 0/35 - Loss: 1.7964
  Batch 15/35 - Loss: 1.7974
  Batch 30/35 - Loss: 1.8044
Avg loss: 

In [35]:
# Release GPU memory
import gc

# 1. Delete large objects
del model_0
del optimizer
del loss_fn

# 2. Run garbage collection (CPU side)
gc.collect()

# 3. Clear CUDA cache (GPU side)
torch.cuda.empty_cache()

## Model with Non-Linearity (Second Attempt: Adding Non-Linearity)

The linear model probably performed poorly (as expected). Adding ReLU activations between layers should help capture more complex patterns. Same architecture otherwise - just want to see what activation functions alone give us. Still flattening the images though, so we're losing all spatial information.

In [36]:
class NonLinearBaseline(nn.Module):
    def __init__(self, in_features, hidden_units, out_features) -> None:
        super().__init__()

        self.layer_stack = nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_features=in_features,
                      out_features=hidden_units),
            nn.ReLU(),
            nn.Linear(in_features=hidden_units,
                      out_features=hidden_units),
            nn.ReLU(),
            nn.Linear(in_features=hidden_units,
                      out_features=out_features)
        )

    
    def forward(self, X):
        return self.layer_stack(X)

In [37]:
model_1 = NonLinearBaseline(in_features=224*224*3,
                            hidden_units=50,
                            out_features=len(pcb_dataset.classes)).to(device=device)

model_1

NonLinearBaseline(
  (layer_stack): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=150528, out_features=50, bias=True)
    (2): ReLU()
    (3): Linear(in_features=50, out_features=50, bias=True)
    (4): ReLU()
    (5): Linear(in_features=50, out_features=6, bias=True)
  )
)

In [38]:
optimizer = SGD(params=model_1.parameters(), lr= 0.01)
loss_fn   = CrossEntropyLoss()

In [39]:
def accuracy_fn(y_pred, y_true):
    correct  = torch.eq(y_pred, y_true).sum().item() 
    accuracy = (correct/len(y_true))*100

    return accuracy

In [40]:
def train_step(dataloader,
               model,
               optimizer,
               accuracy_fn,
               loss_fn):
    
    train_loss, train_acc = 0, 0
    model.to(device)
    for batch, (X, y) in enumerate(dataloader):
        batch += 1
        # Device Agnostic
        X, y = X.to(device), y.to(device)
        #1. Forward pass
        y_pred = model(X)

        #2. Calculate the loss
        loss        = loss_fn(y_pred, y)
        train_loss += loss
        train_acc  +=  accuracy_fn(y_pred=y_pred.argmax(dim=1),
                                   y_true=y)
        
        #3. Optimizer zero grad
        optimizer.zero_grad()

        #4. Backpropagation
        loss.backward()
    
        #5. Optimizer step
        optimizer.step()
        if (batch + 1) % 10 == 0:
            seen = (batch + 1) * X.size(0)
            total = len(dataloader.dataset)

            seen = min(seen, total)

            print(f"Looked at {seen}/{total} samples")
        

    train_loss /= len(dataloader)
    train_acc  /= len(dataloader)


    return train_acc, train_loss

def test_step(model,
              dataloader,
              loss_fn,
              accuracy_fn):
    
    model.to(device)
    test_acc, test_loss  = 0, 0

    with torch.inference_mode():
        for X, y in dataloader:
            X, y   = X.to(device), y.to(device)
            y_pred = model(X)

            test_loss += loss_fn(y_pred, y)
            test_acc  += accuracy_fn(y_pred=y_pred.argmax(dim=1), y_true= y)
        
        test_acc  /= len(dataloader)
        test_loss /= len(dataloader)

    return test_acc, test_loss

In [ ]:
torch.manual_seed(42)

# Measure time
from timeit import default_timer as timer
train_time_start = timer()

Epochs = 10
for epoch in tqdm(range(Epochs)):
    print(f"Epoch: {epoch + 1}\n---------")
    train_acc, train_loss = train_step(dataloader=train_dataloader, 
                                        model=model_1, 
                                        loss_fn=loss_fn,
                                        optimizer=optimizer,
                                        accuracy_fn=accuracy_fn
                                    )
    test_acc, test_loss = test_step(dataloader=test_dataloader,
                                    model=model_1,
                                    loss_fn=loss_fn,
                                    accuracy_fn=accuracy_fn
                                )

    print(f"Train Loss: {train_loss:.3f} | Train Accuracy: {train_acc:.3f} | Test Loss: {test_loss:.3f} | Test Accuracy: {test_acc:.3f}")
train_time_end = timer()
total_train_time_model_1 = print_train_time(start=train_time_start,
                                            end=train_time_end,
                                            device=device)

  0%|          | 0/10 [00:00<?, ?it/s]

Epoch: 1
---------
Looked at 160/554 samples
Looked at 320/554 samples
Looked at 480/554 samples
Train Loss: 1.804 | Train Accuracy: 15.929 | Test Loss: 1.792 | Test Accuracy: 16.288
Epoch: 2
---------
Looked at 160/554 samples
Looked at 320/554 samples
Looked at 480/554 samples
Train Loss: 1.794 | Train Accuracy: 16.643 | Test Loss: 1.792 | Test Accuracy: 16.288
Epoch: 3
---------
Looked at 160/554 samples
Looked at 320/554 samples
Looked at 480/554 samples
Train Loss: 1.793 | Train Accuracy: 15.107 | Test Loss: 1.795 | Test Accuracy: 16.288
Epoch: 4
---------
Looked at 160/554 samples


In [ ]:
# Release GPU memory
import gc

# 1. Delete large objects
del model_1
del optimizer
del loss_fn

# 2. Run garbage collection (CPU side)
gc.collect()

# 3. Clear CUDA cache (GPU side)
torch.cuda.empty_cache()

## A CNN Model (TinyVGG Inspired)

In [ ]:
class CustomCNN(nn.Module):
    def __init__(self, input_shape, hidden_units, output_shape):
        super().__init__()

        self.block_1 = nn.Sequential(
            nn.Conv2d(in_channels=input_shape,
                      out_channels=hidden_units,
                      kernel_size=3,
                      stride=1,
                      padding=1),
            nn.ReLU(),
            nn.Conv2d(in_channels=hidden_units,
                      out_channels=hidden_units,
                      kernel_size=3,
                      stride=1,
                      padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2,
                         stride=2)
        )

        self.block_2 = nn.Sequential(
            nn.Conv2d(in_channels=hidden_units,
                      out_channels=hidden_units,
                      kernel_size=3,
                      stride=1,
                      padding=1),
            nn.ReLU(),
            nn.Conv2d(in_channels=hidden_units,
                      out_channels=hidden_units,
                      kernel_size=3,
                      stride=1,
                      padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2,
                         stride=2)
        )
        
        self.block_3 = nn.Sequential(
            nn.Conv2d(in_channels=hidden_units,
                      out_channels=hidden_units,
                      kernel_size=3,
                      stride=1,
                      padding=1),
            nn.ReLU(),
            nn.Conv2d(in_channels=hidden_units,
                      out_channels=hidden_units,
                      kernel_size=3,
                      stride=1,
                      padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2,
                         stride=2)
        )

        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(in_features=hidden_units,
                      out_features=output_shape)
        )

    def forward(self, x):
        x = self.block_1(x)
        x = self.block_2(x)
        x = self.block_3(x)
        x = self.classifier(x)
        return x

In [ ]:
model_2 = CustomCNN(input_shape=3,
                    hidden_units=10*7*7,
                    output_shape=len(class_name)).to(device)

model_2

In [ ]:
images = torch.randn(size=(32, 3, 224, 224))
test_image = images[0]

test_image.shape

print(f"Image batch shape: {images.shape} -> [batch_size, color_channels, height, width]")
print(f"Single image shape: {test_image.shape} -> [color_channels, height, width]") 
print(f"Single image pixel values:\n{test_image}")

## Summary

The CNN approach should significantly outperform the baseline models. Key observations from this exploration:
- Dataset is small but balanced (693 images, 6 classes)
- Data augmentation is critical given limited samples
- The progression from linear → non-linear → CNN shows the importance of architecture choice
- Next step would be trying transfer learning (ResNet) or moving to YOLO for object detection since we have bounding box annotations available

In [ ]:
loss_fn   = CrossEntropyLoss()
optimizer = SGD(params=model_2.parameters(), lr=0.01)

def accuracy_fn(y_pred, y_true):
    correct  = torch.eq(y_pred,y_true).sum().item()
    accuracy = (correct/len(y_true))*100

    return accuracy

In [ ]:
torch.manual_seed(42)

# Measure time
from timeit import default_timer as timer
from tqdm.auto import tqdm
train_time_start = timer()

Epochs = 10
for epoch in tqdm(range(Epochs)):
    print(f"Epoch: {epoch + 1}\n---------")
    train_acc, train_loss = train_step(dataloader=train_dataloader, 
                                        model=model_2, 
                                        loss_fn=loss_fn,
                                        optimizer=optimizer,
                                        accuracy_fn=accuracy_fn
                                    )
    test_acc, test_loss = test_step(dataloader=test_dataloader,
                                    model=model_2,
                                    loss_fn=loss_fn,
                                    accuracy_fn=accuracy_fn
                                )

    print(f"Train Loss: {train_loss:.3f} | Train Accuracy: {train_acc:.3f} | Test Loss: {test_loss:.3f} | Test Accuracy: {test_acc:.3f}")
train_time_end = timer()
total_train_time_model_1 = print_train_time(start=train_time_start,
                                            end=train_time_end,
                                            device=device)